# 01 - Translation run

The one notebook that calls an LLM. It drives the generate, validate, and fix loop of sql2graph over the evaluation matrix (three target languages, five model configurations, and the 15 gold SQL queries, for 225 translation tasks) and writes one records file per matrix cell to `eval/outputs/records/` (`records_ldbc_<target>_<model>.json`). Every downstream metric notebook reads those records.

The run is resumable: for each cell it fetches the records already on disk and translates only the missing query ids. Pass `force=True` to a `translate(...)` call to re-translate a cell from scratch, or `subset=(...)` to restrict it to specific queries. The five configurations, in the order used everywhere, are `llama3.2`, `qwen3-coder`, `gemma4`, `opus-4-8`, and `opus-4-8-thinking` (the frontier model with extended thinking on); the thinking configuration hits the same API model (`claude-opus-4-8`) but is stratified under its own label. Displayed names are the short tags; the functional ids stay in the run matrix and record filenames.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

import json
import os
from dataclasses import replace

import pandas as pd
from harness import (
    CACHE_DIR,
    DISPLAY_NAME,
    METRICS_DIR,
    RECORDS_DIR,
    RUN_MATRIX,
    billed_input_tokens,
    build_work_items,
    load_dataset,
    load_records,
    mapping_for,
    order_models,
    records_path,
    translate_one,
    write_records,
)

## Helper methods

`translate` spells out the resume logic this notebook is meant to make visible: locate the matrix cell for a (target, model) pair, load its records file, skip query ids already done, and translate only what is missing via `translate_one` (the single-query primitive in the harness). The `force` and `subset` parameters give per-call control over what runs.

In [2]:
RESULT_COLS = ["query_id", "difficulty", "validation_passed", "iterations_used", "status",
               "billed_input_tokens", "output_tokens", "duration_seconds", "error"]


def run_config(target, model):
    """The unique RUN_MATRIX cell for (target, model), keyed on the stratification label."""
    matches = [rc for rc in RUN_MATRIX if rc.target == target and (rc.label or rc.model) == model]
    assert len(matches) == 1, f"{len(matches)} matrix cell(s) for {target}/{model}, expected 1"
    return matches[0]


def translate(target, model, subset=None, force=False):
    """Translate one matrix cell, resuming from disk unless force=True; return its per-query table.

    Set force=True to re-translate the cell from scratch, or subset=("ldbc_q01", ...) to restrict
    the run to specific query ids.
    """
    rc = run_config(target, model)
    if subset is not None:
        rc = replace(rc, subset=subset)
    path = records_path(rc)
    existing = [] if force else (json.loads(path.read_text()) if path.exists() else [])
    done = {r["query_id"] for r in existing}
    if existing:
        print(f"Resume: {len(existing)} record(s) on disk for {path.name}; {len(done)} done.")
    todo = [w for w in build_work_items(rc) if w.query_id not in done]
    print(f"{target}/{model}: {len(todo)} item(s) to translate" + (" (force)" if force else ""))
    mapping = mapping_for(rc.dataset)
    records = list(existing)
    for item in todo:
        records.append(translate_one(rc, item, mapping))
        write_records(path, records)  # incremental: crash-safe
    df = pd.DataFrame(records)
    if not len(df):
        print("No records.")
        return df
    df["billed_input_tokens"] = billed_input_tokens(df["input_tokens"], df["cache_read_tokens"], df["cache_creation_tokens"])
    if "thinking_used" not in df.columns:
        df["thinking_used"] = False
    cols = RESULT_COLS + (["thinking_used"] if "thinking" in str(model) else [])
    return df[cols].sort_values("query_id").reset_index(drop=True)


def summary_by_model(target):
    """Aggregate this target's records on disk, by model (canonical order)."""
    df = pd.DataFrame(load_records(RECORDS_DIR, target=target))
    if not len(df):
        print(f"No records for {target} yet.")
        return None
    df["billed_input_tokens"] = billed_input_tokens(df["input_tokens"], df["cache_read_tokens"], df["cache_creation_tokens"])
    df["model"] = df["model"].map(lambda m: DISPLAY_NAME.get(m, m))  # short display tags
    df["model"] = pd.Categorical(df["model"], order_models(df["model"].unique()), ordered=True)
    return df.groupby("model", observed=True).agg(
        n=("query_id", "count"), passed=("validation_passed", "sum"),
        mean_iterations=("iterations_used", "mean"),
        in_tok=("billed_input_tokens", "sum"), out_tok=("output_tokens", "sum"),
        mean_duration_s=("duration_seconds", "mean"))

## What we are testing

The evaluation matrix (3 targets by 5 model configurations) and the gold query catalogue (the 15 SQL queries by difficulty and SQL feature). This describes the experiment; the Prerequisites section below checks the preconditions for running it.

In [3]:
print(f"{len(RUN_MATRIX)} matrix cell(s) (3 targets x 5 model configs):")
for rc in RUN_MATRIX:
    tag = f" [{rc.label}]" if rc.label else ""
    print(f"  - {rc.dataset} x {rc.target} x {rc.model}{tag} ({rc.provider})")

15 matrix cell(s) (3 targets x 5 model configs):
  - ldbc x cypher x llama3.2:latest (ollama)
  - ldbc x cypher x qwen3-coder:30b (ollama)
  - ldbc x cypher x gemma4:26b (ollama)
  - ldbc x cypher x claude-opus-4-8 (anthropic)
  - ldbc x cypher x claude-opus-4-8 [claude-opus-4-8-thinking] (anthropic)
  - ldbc x aql x llama3.2:latest (ollama)
  - ldbc x aql x qwen3-coder:30b (ollama)
  - ldbc x aql x gemma4:26b (ollama)
  - ldbc x aql x claude-opus-4-8 (anthropic)
  - ldbc x aql x claude-opus-4-8 [claude-opus-4-8-thinking] (anthropic)
  - ldbc x gremlin x llama3.2:latest (ollama)
  - ldbc x gremlin x qwen3-coder:30b (ollama)
  - ldbc x gremlin x gemma4:26b (ollama)
  - ldbc x gremlin x claude-opus-4-8 (anthropic)
  - ldbc x gremlin x claude-opus-4-8 [claude-opus-4-8-thinking] (anthropic)


In [4]:
# Gold query catalogue: what the 15 SQL queries actually are, sorted by feature.
rows = [{"id": q.id, "difficulty": q.difficulty,
         "features": ", ".join(q.sql_features) or "(none)",
         "targets": ", ".join(sorted(q.expected)),
         "description": q.description}
        for q in load_dataset("ldbc")]
catalogue = pd.DataFrame(rows).sort_values(["features", "id"]).reset_index(drop=True)
counts = catalogue["difficulty"].value_counts().reindex(["easy", "medium", "hard"]).to_dict()
print(f"{len(catalogue)} gold queries; difficulty mix: {counts}")
catalogue

15 gold queries; difficulty mix: {'easy': 3, 'medium': 4, 'hard': 8}


,id,difficulty,features,targets,description
0,ldbc_q01,easy,(none),"aql, cypher, gremlin","Point lookup by primary key (Person profile, c..."
1,ldbc_q03,easy,(none),"aql, cypher, gremlin",Lookup persons by first name.
2,ldbc_q14,medium,"aggregation, join","aql, cypher, gremlin",Persons per country via place -> partof -> cou...
3,ldbc_q04,hard,"aggregation, join, order_limit","aql, cypher, gremlin",Top forums by member count (junction-table agg...
4,ldbc_q10,hard,"aggregation, join, order_limit","aql, cypher, gremlin","Optional traversal (LEFT JOIN), forums and the..."
5,ldbc_q11,hard,"aggregation, join, order_limit","aql, cypher, gremlin",HAVING: persons with more than 5 friends.
6,ldbc_q05,hard,"aggregation, join, order_limit, subquery, union","aql, cypher, gremlin",Most-used tags across all posts and comments.
7,ldbc_q15,hard,"cte, join, order_limit, union","aql, cypher, gremlin",Transitive ancestor walk up the Place hierarch...
8,ldbc_q08,hard,"distinct, join","aql, cypher, gremlin",Friends-of-friends who share a tag interest (4...
9,ldbc_q12,hard,"distinct, join, union","aql, cypher, gremlin",UNION: persons who either created OR liked any...


## Prerequisites

Create the output directories and check that the Ollama models are pulled and the Anthropic key is set. Ollama liveness is the hard precondition for the local cells; the database checks live in `05_execution_metrics.ipynb`, the only notebook that needs them.

In [5]:
for d in (RECORDS_DIR, METRICS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

import ollama

ollama_cells = [rc for rc in RUN_MATRIX if rc.provider == "ollama"]
try:
    available = [m.model for m in ollama.Client().list().models]
    print(f"Ollama reachable: {len(available)} model(s) available.")
    for rc in ollama_cells:
        ok = rc.model in available
        print(f"  {rc.model}: " + ("OK" if ok else f"MISSING, run: ollama pull {rc.model}"))
except Exception as exc:
    print(f"WARNING: could not reach Ollama (OLLAMA_HOST unset, SDK default): {type(exc).__name__}: {exc}")
    print("Start it with `ollama serve` before running the translation cells.")

if any(rc.provider == "anthropic" for rc in RUN_MATRIX):
    print("Anthropic: ANTHROPIC_API_KEY " + ("set." if os.environ.get("ANTHROPIC_API_KEY") else "NOT set, the opus rows will error."))

Start it with `ollama serve` before running the translation cells.
Anthropic: ANTHROPIC_API_KEY NOT set, the opus rows will error.


## SQL to Cypher

### llama3.2:latest

In [6]:
translate('cypher', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_cypher_llama3.2_latest.json; 15 done.
cypher/llama3.2:latest: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2404,35,6.945831,None
1,ldbc_q02,easy,True,3,success,8910,162,13.532050,None
2,ldbc_q03,easy,True,1,success,2401,185,8.171838,None
3,ldbc_q04,hard,True,1,success,3172,193,9.199575,None
4,ldbc_q05,hard,True,1,success,3461,81,4.061995,None
5,ldbc_q06,medium,True,1,success,2732,69,3.304163,None
6,ldbc_q07,medium,True,1,success,3199,104,5.110439,None
7,ldbc_q08,hard,True,3,success,9238,421,20.374309,None
8,ldbc_q09,medium,True,1,success,2821,85,3.901266,None
9,ldbc_q10,hard,True,1,success,3173,181,8.464310,None


### qwen3-coder:30b

In [7]:
translate('cypher', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_cypher_qwen3-coder_30b.json; 15 done.
cypher/qwen3-coder:30b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2406,31,17.392788,None
1,ldbc_q02,easy,True,1,success,2890,71,19.496970,None
2,ldbc_q03,easy,True,1,success,2401,22,1.185505,None
3,ldbc_q04,hard,True,1,success,3175,49,3.422444,None
4,ldbc_q05,hard,True,1,success,3464,83,4.775432,None
5,ldbc_q06,medium,True,1,success,2734,47,3.332481,None
6,ldbc_q07,medium,True,1,success,3232,98,5.407516,None
7,ldbc_q08,hard,True,1,success,2841,93,5.113587,None
8,ldbc_q09,medium,True,1,success,2831,65,3.474695,None
9,ldbc_q10,hard,True,1,success,3175,42,4.065331,None


### gemma4:26b

In [8]:
translate('cypher', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_cypher_gemma4_26b.json; 15 done.
cypher/gemma4:26b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2735,422,37.736048,None
1,ldbc_q02,easy,True,1,success,3262,1215,56.467919,None
2,ldbc_q03,easy,True,1,success,2730,415,5.185734,None
3,ldbc_q04,hard,True,1,success,3563,1024,12.430539,None
4,ldbc_q05,hard,True,1,success,3864,2099,24.740709,None
5,ldbc_q06,medium,True,1,success,3094,1236,15.012691,None
6,ldbc_q07,medium,True,1,success,3642,1904,22.617156,None
7,ldbc_q08,hard,True,1,success,3222,2134,24.555018,None
8,ldbc_q09,medium,True,1,success,3203,2013,23.373235,None
9,ldbc_q10,hard,True,1,success,3566,828,10.780821,None


### claude-opus-4-8

In [9]:
translate('cypher', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_cypher_claude-opus-4-8.json; 15 done.
cypher/claude-opus-4-8: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3825,50,2.084392,None
1,ldbc_q02,easy,True,1,success,4566,87,7.168288,None
2,ldbc_q03,easy,True,1,success,3823,45,1.633420,None
3,ldbc_q04,hard,True,1,success,5186,76,2.711763,None
4,ldbc_q05,hard,True,1,success,5709,96,2.775870,None
5,ldbc_q06,medium,True,1,success,4385,73,2.871345,None
6,ldbc_q07,medium,True,1,success,5152,140,2.847695,None
7,ldbc_q08,hard,True,1,success,4592,151,2.855862,None
8,ldbc_q09,medium,True,1,success,4572,118,3.690251,None
9,ldbc_q10,hard,True,1,success,5188,89,2.324540,None


### claude-opus-4-8-thinking

In [10]:
translate('cypher', 'claude-opus-4-8-thinking')

Resume: 15 record(s) on disk for records_ldbc_cypher_claude-opus-4-8-thinking.json; 15 done.
cypher/claude-opus-4-8-thinking: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error,thinking_used
0,ldbc_q01,easy,True,1,success,3899,50,3.011928,None,False
1,ldbc_q02,easy,True,1,success,4640,109,3.088816,None,True
2,ldbc_q03,easy,True,1,success,3897,51,2.015992,None,False
3,ldbc_q04,hard,True,1,success,5260,76,2.394907,None,False
4,ldbc_q05,hard,True,1,success,5783,671,9.073131,None,True
5,ldbc_q06,medium,True,1,success,4459,73,2.110404,None,False
6,ldbc_q07,medium,True,1,success,5226,140,4.997396,None,False
7,ldbc_q08,hard,True,1,success,4666,379,4.912796,None,True
8,ldbc_q09,medium,True,1,success,4645,213,3.678660,None,True
9,ldbc_q10,hard,True,1,success,5262,83,2.121678,None,False


### Aggregation by model

In [11]:
summary_by_model('cypher')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
llama3.2:latest,15,15,1.266667,56264,2148,8.117859
qwen3-coder:30b,15,15,1.000000,43926,968,8.400243
gemma4:26b,15,15,1.000000,49506,25596,32.368910
claude-opus-4-8,15,15,1.000000,70926,1413,2.960200
claude-opus-4-8-thinking,15,15,1.000000,72035,3150,4.107374


## SQL to AQL

### llama3.2:latest

In [12]:
translate('aql', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_aql_llama3.2_latest.json; 15 done.
aql/llama3.2:latest: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3084,110,10.834274,None
1,ldbc_q02,easy,True,1,success,3412,163,8.007973,None
2,ldbc_q03,easy,True,1,success,3081,124,5.935551,None
3,ldbc_q04,hard,True,2,success,8603,329,16.883880,None
4,ldbc_q05,hard,True,2,success,9219,333,17.596494,None
5,ldbc_q06,medium,True,1,success,3462,198,9.611270,None
6,ldbc_q07,medium,True,1,success,3806,213,10.314666,None
7,ldbc_q08,hard,True,3,success,12952,746,37.362195,None
8,ldbc_q09,medium,False,3,max_iterations_reached,11500,465,23.329497,None
9,ldbc_q10,hard,True,2,success,8779,555,28.752213,None


### qwen3-coder:30b

In [13]:
translate('aql', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_aql_qwen3-coder_30b.json; 15 done.
aql/qwen3-coder:30b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3088,49,18.903891,None
1,ldbc_q02,easy,True,1,success,3441,61,3.055459,None
2,ldbc_q03,easy,True,1,success,3083,35,1.827960,None
3,ldbc_q04,hard,True,1,success,4018,88,6.200545,None
4,ldbc_q05,hard,True,1,success,4334,93,5.246304,None
5,ldbc_q06,medium,True,1,success,3466,55,4.275656,None
6,ldbc_q07,medium,True,1,success,3835,104,6.019845,None
7,ldbc_q08,hard,True,1,success,3601,113,6.641182,None
8,ldbc_q09,medium,True,1,success,3623,79,4.078746,None
9,ldbc_q10,hard,True,1,success,4018,51,4.485285,None


### gemma4:26b

In [14]:
translate('aql', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_aql_gemma4_26b.json; 15 done.
aql/gemma4:26b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3441,453,21.159219,None
1,ldbc_q02,easy,True,1,success,3818,1225,14.435183,None
2,ldbc_q03,easy,True,1,success,3436,429,5.401434,None
3,ldbc_q04,hard,True,1,success,4432,1981,24.457997,None
4,ldbc_q05,hard,True,1,success,4764,6385,72.695676,None
5,ldbc_q06,medium,True,1,success,3856,1801,22.087202,None
6,ldbc_q07,medium,True,1,success,4258,2302,26.812581,None
7,ldbc_q08,hard,True,2,success,8617,11571,133.419983,None
8,ldbc_q09,medium,True,1,success,4026,5565,64.891218,None
9,ldbc_q10,hard,True,1,success,4435,1447,18.305811,None


### claude-opus-4-8

In [15]:
translate('aql', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_aql_claude-opus-4-8.json; 15 done.
aql/claude-opus-4-8: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,5049,77,3.278285,None
1,ldbc_q02,easy,True,1,success,5619,80,2.613209,None
2,ldbc_q03,easy,True,1,success,5047,62,4.596436,None
3,ldbc_q04,hard,True,1,success,6693,104,2.334190,None
4,ldbc_q05,hard,True,1,success,7275,212,3.771293,None
5,ldbc_q06,medium,True,1,success,5682,100,2.337970,None
6,ldbc_q07,medium,True,1,success,6278,142,2.853055,None
7,ldbc_q08,hard,True,1,success,5944,216,2.959778,None
8,ldbc_q09,medium,True,1,success,5971,142,2.539799,None
9,ldbc_q10,hard,True,1,success,6695,95,2.238497,None


### claude-opus-4-8-thinking

In [16]:
translate('aql', 'claude-opus-4-8-thinking')

Resume: 15 record(s) on disk for records_ldbc_aql_claude-opus-4-8-thinking.json; 15 done.
aql/claude-opus-4-8-thinking: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error,thinking_used
0,ldbc_q01,easy,True,1,success,5123,87,2.845516,None,True
1,ldbc_q02,easy,True,1,success,5693,88,2.187544,None,True
2,ldbc_q03,easy,True,1,success,5121,70,2.842752,None,True
3,ldbc_q04,hard,True,1,success,6767,132,3.167358,None,True
4,ldbc_q05,hard,True,1,success,7349,878,11.965075,None,True
5,ldbc_q06,medium,True,1,success,5756,180,3.965427,None,True
6,ldbc_q07,medium,True,1,success,6352,212,3.365681,None,True
7,ldbc_q08,hard,True,1,success,6018,733,8.686078,None,True
8,ldbc_q09,medium,True,1,success,6045,276,4.286678,None,True
9,ldbc_q10,hard,True,1,success,6769,155,4.280975,None,True


### Aggregation by model

In [17]:
summary_by_model('aql')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
llama3.2:latest,15,12,1.800000,109403,5193,21.446847
qwen3-coder:30b,15,15,1.000000,55295,1152,7.132720
gemma4:26b,15,15,1.066667,65868,46451,38.112693
claude-opus-4-8,15,15,1.000000,91382,1913,3.060915
claude-opus-4-8-thinking,15,15,1.000000,92492,4823,5.175006


## SQL to Gremlin

### llama3.2:latest

In [18]:
translate('gremlin', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_gremlin_llama3.2_latest.json; 15 done.
gremlin/llama3.2:latest: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2671,184,13.159977,None
1,ldbc_q02,easy,False,3,max_iterations_reached,9857,662,30.228762,None
2,ldbc_q03,easy,True,1,success,2668,164,7.280943,None
3,ldbc_q04,hard,False,3,max_iterations_reached,11898,725,36.715658,None
4,ldbc_q05,hard,False,3,max_iterations_reached,12070,820,41.049272,None
5,ldbc_q06,medium,False,3,max_iterations_reached,9717,369,17.293283,None
6,ldbc_q07,medium,False,3,max_iterations_reached,11101,758,36.326841,None
7,ldbc_q08,hard,True,1,success,3183,698,32.553665,None
8,ldbc_q09,medium,False,3,max_iterations_reached,11551,792,37.760254,None
9,ldbc_q10,hard,False,3,max_iterations_reached,11047,613,29.893580,None


### qwen3-coder:30b

In [19]:
translate('gremlin', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_gremlin_qwen3-coder_30b.json; 15 done.
gremlin/qwen3-coder:30b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2675,56,11.609653,NaN
1,ldbc_q02,easy,True,1,success,2992,72,2.872458,NaN
2,ldbc_q03,easy,True,1,success,2670,41,1.913921,NaN
3,ldbc_q04,hard,True,3,success,10842,340,18.067965,NaN
4,ldbc_q05,hard,True,1,success,3670,109,5.568143,NaN
5,ldbc_q06,medium,True,1,success,3045,58,3.427081,NaN
6,ldbc_q07,medium,True,1,success,3377,93,4.488420,NaN
7,ldbc_q08,hard,False,1,generation_hang,0,0,NaN,ollama generation hung on this prompt in three...
8,ldbc_q09,medium,True,1,success,3203,108,5.338304,NaN
9,ldbc_q10,hard,True,1,success,3387,94,5.841224,NaN


### gemma4:26b

In [20]:
translate('gremlin', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_gremlin_gemma4_26b.json; 15 done.
gremlin/gemma4:26b: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2998,1239,24.769073,None
1,ldbc_q02,easy,True,1,success,3322,829,10.268602,None
2,ldbc_q03,easy,True,1,success,2993,697,8.568272,None
3,ldbc_q04,hard,True,1,success,3758,1962,23.684939,None
4,ldbc_q05,hard,True,1,success,4055,5006,58.420987,None
5,ldbc_q06,medium,True,1,success,3393,2372,28.340513,None
6,ldbc_q07,medium,True,1,success,3738,2995,34.905015,None
7,ldbc_q08,hard,True,1,success,3556,3696,50.942750,None
8,ldbc_q09,medium,True,1,success,3570,4642,63.552989,None
9,ldbc_q10,hard,True,1,success,3761,2985,35.526123,None


### claude-opus-4-8

In [21]:
translate('gremlin', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_gremlin_claude-opus-4-8.json; 15 done.
gremlin/claude-opus-4-8: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,4127,76,2.570988,None
1,ldbc_q02,easy,True,1,success,4571,85,1.942554,None
2,ldbc_q03,easy,True,1,success,4125,61,2.109440,None
3,ldbc_q04,hard,True,1,success,5261,78,2.800566,None
4,ldbc_q05,hard,True,1,success,5724,78,4.037739,None
5,ldbc_q06,medium,True,1,success,4683,81,2.037096,None
6,ldbc_q07,medium,True,1,success,5153,186,4.283523,None
7,ldbc_q08,hard,True,1,success,4926,141,4.017482,None
8,ldbc_q09,medium,True,1,success,4933,157,3.045629,None
9,ldbc_q10,hard,True,1,success,5263,73,2.142205,None


### claude-opus-4-8-thinking

In [22]:
translate('gremlin', 'claude-opus-4-8-thinking')

Resume: 15 record(s) on disk for records_ldbc_gremlin_claude-opus-4-8-thinking.json; 15 done.
gremlin/claude-opus-4-8-thinking: 0 item(s) to translate


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error,thinking_used
0,ldbc_q01,easy,True,1,success,4201,76,2.924066,None,False
1,ldbc_q02,easy,True,1,success,4645,95,2.401232,None,True
2,ldbc_q03,easy,True,1,success,4199,61,2.684893,None,False
3,ldbc_q04,hard,True,1,success,5335,477,8.253819,None,True
4,ldbc_q05,hard,True,1,success,5798,715,10.829722,None,True
5,ldbc_q06,medium,True,1,success,4757,483,7.049105,None,True
6,ldbc_q07,medium,True,1,success,5227,355,5.906375,None,True
7,ldbc_q08,hard,True,1,success,5000,1640,19.971580,None,True
8,ldbc_q09,medium,True,1,success,5007,715,9.505123,None,True
9,ldbc_q10,hard,True,1,success,5337,298,4.775298,None,True


### Aggregation by model

In [23]:
summary_by_model('gremlin')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
llama3.2:latest,15,3,2.600000,140396,9424,30.926102
qwen3-coder:30b,15,11,1.400000,64911,2006,8.829270
gemma4:26b,15,15,1.066667,57192,46426,38.251002
claude-opus-4-8,15,15,1.000000,73826,1383,2.732615
claude-opus-4-8-thinking,15,15,1.000000,74936,7279,7.392287


## Run-level summary

In [24]:
df = pd.DataFrame(load_records(RECORDS_DIR))
if len(df):
    df["billed_input_tokens"] = billed_input_tokens(df["input_tokens"], df["cache_read_tokens"], df["cache_creation_tokens"])
    df["model"] = df["model"].map(lambda m: DISPLAY_NAME.get(m, m))  # short display tags
    df["model"] = pd.Categorical(df["model"], order_models(df["model"].unique()), ordered=True)
    print(f"Total records: {len(df)}")
    print(f"Validation passed: {int(df['validation_passed'].sum())} / {len(df)}")
    print(f"Total tokens: {int(df['billed_input_tokens'].sum()):,} in / {int(df['output_tokens'].sum()):,} out")
    print(f"Mean duration: {df['duration_seconds'].mean():.2f}s")
    display(df.groupby(["target", "model"], observed=True).agg(
        n=("query_id", "count"), passed=("validation_passed", "sum"),
        mean_iter=("iterations_used", "mean"),
        in_tok=("billed_input_tokens", "sum"), out_tok=("output_tokens", "sum")))
else:
    print("No records yet.")

Total records: 225
Validation passed: 206 / 225
Total tokens: 1,118,358 in / 159,325 out
Mean duration: 14.65s


n  passed  mean_iter  in_tok  out_tok
target  model                                                           
aql     llama3.2:latest           15      12   1.800000  109403     5193
        qwen3-coder:30b           15      15   1.000000   55295     1152
        gemma4:26b                15      15   1.066667   65868    46451
        claude-opus-4-8           15      15   1.000000   91382     1913
        claude-opus-4-8-thinking  15      15   1.000000   92492     4823
cypher  llama3.2:latest           15      15   1.266667   56264     2148
        qwen3-coder:30b           15      15   1.000000   43926      968
        gemma4:26b                15      15   1.000000   49506    25596
        claude-opus-4-8           15      15   1.000000   70926     1413
        claude-opus-4-8-thinking  15      15   1.000000   72035     3150
gremlin llama3.2:latest           15       3   2.600000  140396     9424
        qwen3-coder:30b           15      11   1.400000   64911     2006
        gemma4:26b                15      15   1.066667   57192    46426
        claude-opus-4-8           15      15   1.000000   73826     1383
        claude-opus-4-8-thinking  15      15   1.000000   74936     7279

## Summary

This notebook produced the raw translation records, one JSON file per matrix cell, that are the single source of truth for every metric. Nothing here judges quality; it only records what the loop did (validity, iterations, duration, exact token counts) and the query it emitted.

Records written to `outputs/records/`. Proceed to **`02_behavioural_metrics.ipynb`**.